In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import pyplot

import os, sys
sys.path.append(os.path.split(os.getcwd())[0]+'/src/')
sys.path.append(os.path.split(os.getcwd())[0]+'/src/magnus')

# import magnus
import magnus.magnus as magnus
import magnus.oscprob.oscprob as oscprob
import magnus.oscprob.oscprobstd as oscprobstd
import magnus.hamiltonians.hamiltonians2nu as hamiltonians2nu
import magnus.hamiltonians.hamiltonians3nu as hamiltonians3nu
import magnus.matter as matter
import magnus.globaldefs as gd

To compute neutrino oscillation proabilities, Mag$\nu$s needs only three ingredients: 
* a Hamiltonian (`H_func`), written in the flavor basis, as a function of neutrino position (or time),
* the initial neutrino position (or time, `t_ini`), and
* the final neutrino position (or time, `t_fin`).

In practice, in most scenarios the Hamiltonian depends on the neutrino energy, whose value we often vary, so the neutrino energy becomes effectively also a free parameter to vary.

Let us start by computing oscillation probabilities in vacuum in a two-neutrino system. Although the user can provide their own custom Hamiltonian, Mag$\nu$s conveniently provides a library of commonly used Hamiltonians for two-neutrino and three-neutrinos oscillations (and also for 3+1 and 3+2 systems, with one and two extra flavors, respectively).

To access the library of two-neutrino Hamiltonians, we import

In [9]:
import magnus.hamiltonians.hamiltonians2nu as hamiltonians2nu

We will use the two-neutrino Hamiltonian in vacuum, i.e.,

\begin{equation}
\mathbf{H}_{2\nu}^{\rm vac}
=
\mathbf{U}^T 
\left(
 \begin{array}{cc}
  \frac{\Delta m^2}{2E} & 0 \\
  0 & -\frac{\Delta m^2}{2E} \\
 \end{array}
\right)
\mathbf{U} \;,
\end{equation}

where the mixing matrix, $\mathbf{U}$, is parametrized by a single mixing angle, $\theta$, i.e.,

\begin{equation}
 \mathbf{U}
 =
 \left(
  \begin{array}{cc}
   \cos\theta  & \sin\theta \\
   -\sin\theta & \cos\theta
  \end{array}
 \right) \;.
\end{equation}

We need values for the mixing parameters $\theta$ and $\Delta m^2$.  Users can provide any values they wish.  For this example, we will shows oscillations between $\nu_e$ and $\nu_mu$, and use central values of the $\theta_{12}$ and $\Delta m_{21}^2$ parameters from the NuFit 6.0 global fit to oscillation data, which are predefined in the Mag$\nu$s globaldefs module, i.e.,

In [49]:
import magnus.globaldefs as gd

sth = gd.S12_NO_BF_NUFIT_6_0 # sin(theta) [adim]
Dm2 = gd.D21_NO_BF_NUFIT_6_0 # [eV^2]

And now we define our Hamiltonian, as a function of neutrino position (`l`) and energy (`energy`).  In this case, the dependence on neutrino position is a dummy dependence, since the Hamiltonian in vacuum is position-independent. However, Mag$\nu$s expects a position-dependent Hamiltonian in general.

In [46]:
def H_2nu_vac(l, energy):
    return hamiltonians2nu.hamiltonian_2nu_vacuum(energy, sth, Dm2)

The oscillation probabilities are computed using the `osc_prob` function of the `oscprob` module, which we import as

In [3]:
import magnus.oscprob.oscprobstd as oscprobstd

Internally, this function performs Magnus expansion to compute the evolution of the neutrino oscillation amplitude from `t_ini` to `t_fin` by partitioning this interval into subintervals, computing the evolution operator inside each subinterval, and performing their time-ordered product. The user manipulates the options of the Magnus expansion indirectly, via parameters passed to osc_prob.

(Users interested in using Mag$\nu$s to compute the Magnus expansion of an arbitrary matrix exponential, not only in neutrino oscillations, should look at the notebook `10_magnus_matrix_exponential.ipynb`.)

The `osc_prob` function computes probabilities for *any* number of neutrino flavors.  If it is fed a $2 \times 2$ Hamiltonian, then it will return probabilities for a two-neutrino system; if it is fed a $3 \times 3$ Hamiltonian, it will return probabilities for a three-neutrino system, *etc*.

Let's compute the probability for a baseline of 10 km and an energy of 1 MeV.

In [42]:
baseline = 10.*gd.CONV_KM_TO_INV_EV # 10 km in eV^{-1} [eV^{-1}]
energy = 1.*gd.UNIT_MEV # [eV]

And now we call `osc_prob` to compute the probability from `t_ini = 0.0` to `t_fin = baseline`:

In [47]:
P = oscprob.osc_prob(lambda l: H_2nu_vac(l, energy), 0.0, baseline)
P

array([[0.43678029, 0.56321971],
       [0.56321971, 0.43678029]])

The function `osc_prob` returns a symmetric probability matrix,
\begin{equation}
 \mathbf{P}_{2\nu}
 =
 \left(
  \begin{array}{cc}
   P_{ee}    & P_{e\mu} \\
   P_{\mu e} & P_{\mu\mu}
  \end{array}
 \right) \;,
\end{equation}
where $P_{\mu e} = P_{e \mu}$.

We can select individual probabilities using flavor indices predefined in the `globaldefs` module.  I.e., for $\nu_e \to \nu_e$,

In [38]:
P[gd.NUE][gd.NUE]

0.4367802885331587

and, for $\nu_e \to \nu_\mu$,

In [39]:
P[gd.NUE][gd.NUMU]

0.5632197114668414

By default, `osc_prob` uses fourth-order Magnus expansion (i.e., `magnus_exp_order = 4`).  However, for oscillations in vacuum the Hamiltonian is time-independent, and so the only nonzero term of the Magnus expansion is the first one. *I.e.*, the evolution operator is simply $e^{-i H L}$, with $L = t_{\rm fin}-t_{\rm ini}$). 

So we can obtain the same result for the probability matrix using `magnus_exp_order = 1` instead:

In [43]:
oscprob.osc_prob(lambda l: H_2nu_vac(l, energy), 0.0, baseline, magnus_exp_order=1)

array([[0.43678029, 0.56321971],
       [0.56321971, 0.43678029]])

Let's now move on to three-neutrino oscillations.  For this, we import the predefined three-neutrino Hamiltonians as

In [50]:
import magnus.hamiltonians.hamiltonians3nu as hamiltonians3nu

The vacuum Hamiltonian is now a $3 \times 3$ matrix, i.e.,
\begin{equation}
 \mathbf{H}_{3\nu}^{\rm vac}
 =
 \mathbf{U}_{\rm PMNS}^\dagger
 \left(
  \begin{array}{ccc}
   0 & 0 & 0 \\
   0 & \frac{\Delta m_{21}^2}{2E} & 0 \\
   0 & 0 & \frac{\Delta m_{31}^2}{2E}
  \end{array}
 \right)
 \mathbf{U}_\textrm{PMNS} \;,
\end{equation}
where $\mathbf{U}_{\rm PMNS}$ is the complex-valued Pontecorvo-Maki-Nakagawa-Sakata (PMNS) matrix, parametrized using three mixing angles, $\theta_{12}$, $\theta_{23}$, and $\theta_{13}$, and one CP-violation phase, $\delta_{\rm CP}$.

Like before, for this example we set the values of the mixing parameters to their central values from NuFit 6.0, *i.e.*,

In [52]:
s12 = gd.S12_NO_BF_NUFIT_6_0 # sin(theta_12) [adim]
s23 = gd.S23_NO_BF_NUFIT_6_0 # sin(theta_23) [adim]
s13 = gd.S13_NO_BF_NUFIT_6_0 # sin(theta_13) [adim]
dCP = gd.DCP_NO_BF_NUFIT_6_0 # [radian]
D21 = gd.D21_NO_BF_NUFIT_6_0 # [eV^2]
D31 = gd.D31_NO_BF_NUFIT_6_0 # [eV^2]

Like before, we now define our Hamiltonian function in vacuum, giving it a dummy dependence on the neutrino position,

In [56]:
def H_3nu_vac(l, energy):
    return hamiltonians3nu.hamiltonian_3nu_vacuum(energy, s12, s23, s13, dCP, D21, D31)

And we compute the oscillation probability, as before, using the `osc_prob` function,

In [57]:
oscprob.osc_prob(lambda l: H_3nu_vac(l, energy), 0.0, baseline, magnus_exp_order=1)

array([[0.44464666, 0.29867235, 0.25668099],
       [0.25107128, 0.63934947, 0.10957925],
       [0.30428206, 0.06197818, 0.63373976]])

In this case, the result is the probability matrix
\begin{equation}
 \mathbf{P}_{3\nu}
 =
 \left(
  \begin{array}{ccc}
   P_{ee}     & P_{e\mu}    & P_{e\tau} \\
   P_{\mu e}  & P_{\mu\mu}  & P_{\mu\tau} \\
   P_{\tau e} & P_{\tau\mu} & P_{\tau\tau}
  \end{array}
 \right) \;,
\end{equation}
where $P_{\alpha \beta} = P_{\beta \alpha}$, with $\alpha, \beta = e, \mu, \tau$.

Let's now vary the baseline and energy and plot the two- and three-neutrino probabilities.

Now let's add matter effects.  First, we consider oscillations in matter with uniform density.

In [45]:
oscprob.osc_prob(lambda l: H_2nu(l, energy), 0.0, baseline, magnus_exp_order=1, verbose=2)

.----------------------------------------.
|   __  __                               |
|  |  \/  | __ _  __ _ _ __  _   _ ___   |
|  | |\/| |/ _` |/ _` | '_ \| | | / __|  |
|  | |  | | (_| | (_| | | | | |_| \__ \  |
|  |_|  |_|\__,_|\__, |_| |_|\__,_|___/  |
|                |___/                   |
'----------------------------------------'
Version: 0.10

Parameters passed to function magnus.osc_prob in this run:
   H_func = <lambda>
   t_ini = 0.0
   t_fin = 50677300000.0
   n_slabs = 1
   n_tpts_per_slab = 1
   n_tpts_per_slab = None
   magnus_exp_order = 1
   n_jobs = 1
   integration_method = trapezoid
   rtol = 0.001
   atol = 0.001
   growth_factor_n_slabs = 1.5
   growth_factor_n_tpts_per_slab = 1.5
   max_num_loops = 50
   max_n_slabs = 2000
   max_n_tpts_per_slab = 500
   validate_input = True
   save_log = False
   filename_log = ./out.log
   verbose = 2

Running loops until requested rtol and atol are achieved:
   Loop #1:
      n_slabs = 1
      n_tpts_per_slab = 100
   Lo

array([[0.43678029, 0.56321971],
       [0.56321971, 0.43678029]])

Since we will keep these values fixed, we compute $\mathbf{U}_{\rm PMNS}$ once and save it:

In [53]:
U = hamiltonians3nu.pmns_mixing_matrix(s12, s23, s13, dCP)
U

array([[ 0.82260088+0.j        ,  0.54879668+0.j        ,
        -0.12621395+0.07886723j],
       [-0.33205012+0.04497784j,  0.65362818+0.03000688j,
         0.67793031+0.j        ],
       [ 0.45690946+0.04776256j, -0.51930398+0.0318647j ,
         0.71990312+0.j        ]])

(In cases where we want to compute oscillations probabilities for varying values of the mixing parameters---say, in comparison to experimental data---one would need to compute the PMNS matrix for each set of test values of the mixing parameters instead.)